In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.stats import wilcoxon
from scipy.stats import gaussian_kde
import os

In [11]:
# ---- Config ----
CSV_PATH = "../outputs/microexon_final.csv"
OUT_DIR = "../outputs/"
FONT_SIZE = 14

# ---- Colors ----
# Violin fills: grey (whole) + orange (microexon) — matches paired half-violin
C_MICRO_EDGE = "#D55E00"       # dark orange edge
C_MICRO_LIGHT = "#FDD9A0"      # pale orange fill
C_REST_EDGE = "black"
C_REST_LIGHT = "#D9D9D9"       # grey fill

# Connecting line colormap: black -> white -> orange (matches violin border colors)
cmap = LinearSegmentedColormap.from_list(
    "micro_rest", ["#555555", "#FFFFFF", "#D55E00"]
)

# ---- Fonts: Arial, uniform size ----
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "svg.fonttype": "none",
})

# ---- Load data ----
df = pd.read_csv(CSV_PATH)
mini = df["exon_pDP_pDP"].dropna().values
whole = df["protein_pDP_pDP"].dropna().values
n = min(len(mini), len(whole))
mini, whole = mini[:n], whole[:n]
diff = mini - whole

# Stats
stat, p = wilcoxon(mini, whole)
print(f"FuzDrop pDP: microexon vs whole protein (n={n})")
print(f"  Microexon: median={np.median(mini):.4f}, mean={np.mean(mini):.4f}")
print(f"  Whole:     median={np.median(whole):.4f}, mean={np.mean(whole):.4f}")
print(f"  Wilcoxon: stat={stat:.0f}, p={p:.2e}")

# Per-panel normalization for line colors
dmin, dmax = diff.min(), diff.max()
panel_norm = TwoSlopeNorm(vcenter=0, vmin=dmin, vmax=dmax)
order = np.argsort(np.abs(diff))

# ---- Figure ----
fig, ax = plt.subplots(figsize=(7, 4.5), facecolor="white")

# Y positions
y_whole = 0     # bottom half-violin (faces down)
y_micro = 0.3   # top half-violin (faces up)

# --- Compute both KDEs via seaborn (matches distribution plot exactly) ---
# Extract KDE curves from seaborn (same bandwidth, same cut as distribution plot)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _fig_tmp, _ax_tmp = plt.subplots(figsize=(1, 1))
    sns.kdeplot(whole, ax=_ax_tmp, color=C_REST_EDGE)
    _line_w = _ax_tmp.lines[-1]
    x_w_sns, y_w_sns = _line_w.get_xdata(), _line_w.get_ydata()
    plt.close(_fig_tmp)

    _fig_tmp, _ax_tmp = plt.subplots(figsize=(1, 1))
    sns.kdeplot(mini, ax=_ax_tmp, color=C_MICRO_EDGE)
    _line_m = _ax_tmp.lines[-1]
    x_m_sns, y_m_sns = _line_m.get_xdata(), _line_m.get_ydata()
    plt.close(_fig_tmp)

# Use the exact x-range from the distribution plot's auto xlim
x_grid = np.linspace(-0.222, 1.346, 500)

# Interpolate onto shared grid
density_w = np.interp(x_grid, x_w_sns, y_w_sns, left=0, right=0)
density_m = np.interp(x_grid, x_m_sns, y_m_sns, left=0, right=0)

# Scale each violin independently to the same peak width
scale_w = 0.12 / density_w.max()
scale_m = 0.12 / density_m.max()

# --- Half-violin: Whole Protein (faces down, below y=0) ---
ax.fill_between(x_grid, y_whole, y_whole - density_w * scale_w,
                color=C_REST_LIGHT, edgecolor=C_REST_EDGE, linewidth=0.8, zorder=3)
ax.plot(x_grid, y_whole - density_w * scale_w,
        color=C_REST_EDGE, linewidth=0.8, zorder=4)

# --- Half-violin: Microexon (faces up, above y=0.3) ---
ax.fill_between(x_grid, y_micro, y_micro + density_m * scale_m,
                color=C_MICRO_LIGHT, edgecolor=C_MICRO_EDGE, linewidth=0.8, zorder=3)
ax.plot(x_grid, y_micro + density_m * scale_m,
        color=C_MICRO_EDGE, linewidth=0.8, zorder=4)

# --- Paired connecting lines ---
for rank, idx in enumerate(order):
    color = cmap(panel_norm(diff[idx]))
    ax.plot([whole[idx], mini[idx]], [y_whole, y_micro],
            color=color, linewidth=1.0, alpha=0.5, zorder=2 + rank * 0.01)
    ax.scatter([whole[idx], mini[idx]], [y_whole, y_micro],
               s=16, color=color, edgecolor="white", linewidth=0.3,
               zorder=10 + rank * 0.01)

# --- Axes ---
ax.set_yticks([y_whole, y_micro])
ax.set_yticklabels(["Whole\nProtein", "Microexon"])
ax.set_xlabel("FuzDrop pDP")
ax.set_xlim(-0.222, 1.346)
ax.set_ylim(-0.2, 0.55)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=FONT_SIZE, length=4, width=0.8)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=TwoSlopeNorm(vcenter=0, vmin=-1, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.08, fraction=0.03, shrink=0.65)
cbar.set_label("Microexon > Whole  ←  Difference  →  Microexon < Whole",
               fontsize=FONT_SIZE, labelpad=8)
cbar.set_ticks([])
cbar.outline.set_linewidth(0.8)

fig.tight_layout()

path_png = os.path.join(OUT_DIR, "microexon_pdp_halfviolin.png")
path_svg = os.path.join(OUT_DIR, "microexon_pdp_halfviolin.svg")
fig.savefig(path_png, bbox_inches="tight", facecolor="white")
fig.savefig(path_svg, bbox_inches="tight", facecolor="white")
plt.close()
print(f"\nSaved: {path_png}")
print(f"Saved: {path_svg}")

FuzDrop pDP: microexon vs whole protein (n=186)
  Microexon: median=0.4763, mean=0.5274
  Whole:     median=0.4412, mean=0.4592
  Wilcoxon: stat=6809, p=1.03e-02

Saved: ../outputs/microexon_pdp_halfviolin.png
Saved: ../outputs/microexon_pdp_halfviolin.svg


In [10]:
df.columns.tolist()

['Unnamed: 0.4',
 'EVENT',
 'Unnamed: 0.3',
 'Unnamed: 0.2',
 'Unnamed: 0.1',
 'Unnamed: 0',
 'GENE',
 'COORD',
 'LENGTH',
 'Whole_Brain_b',
 'Cortex',
 'Frontal_Gyrus_young',
 'Frontal_Gyrus_old',
 'Sup_Temporal_Gyrus',
 'Cerebellum_a',
 'Cerebellum_c',
 'Neurons',
 'Neurons_Cortex_KCl_0h',
 'Neurons_Cortex_KCl_1h',
 'Neurons_Cortex_KCl_6h',
 'Oocyte_a_A',
 'Zygote_a_A',
 'Embr_2C_a_A',
 'Embr_2C_a_B',
 'Embr_4C_a_A',
 'Embr_4C_a_B',
 'Embr_4C_a_C',
 'Embr_8C_a_A',
 'Embr_8C_a_B',
 'Embr_8C_a_C',
 'Embr_8C_a_D',
 'Embr_Morula_a_A',
 'Embr_Morula_a_B',
 'Embr_ICM_a_A',
 'ESC_H1_a',
 'ESC_H1_b',
 'ESC_H1_c',
 'ESC_H1_d',
 'ESC_H9_a',
 'ESC_H9_b',
 'iPS_a',
 'iPS_b',
 'Embr_TE_a_A',
 'Embr_TE_a_B',
 'HFDPC',
 'MSC',
 'NCC_Cranial_c',
 'NCC_default_b',
 'NCC_Enteric_b',
 'NCC_Melano_biased_b',
 'NPC_a',
 'NPC_b',
 'Astrocytes',
 'Microglia',
 'Oligodendrocytes',
 'Pancreas_Alpha_Old',
 'Pancreas_Alpha_Young',
 'Pancreas_Beta_Old',
 'Pancreas_Beta_Young',
 'Pancreas_Acinar_Old',
 'Pancreas